### LOCAL Qwen3 TTS
## https://github.com/thorstenMueller/Thorsten-Voice/tree/master/Youtube

In [1]:
# Import some dependencies
import torch

!apt-get install sox

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
sox is already the newest version (14.4.2+git20190427-2+deb11u2ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [2]:
# Let's check GPU / CUDA information

if torch.cuda.is_available():
    print("CUDA is available! Here are the GPU details:")
    print(f"  Number of GPUs: {torch.cuda.device_count()}")
    print(f"  Current GPU device: {torch.cuda.current_device()}")
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA Version: {torch.version.cuda}")

    device_properties = torch.cuda.get_device_properties(0)
    print(f"  Total Memory: {device_properties.total_memory / (1024**3):.2f} GB")
    print(f"  Compute Capability: {device_properties.major}.{device_properties.minor}")
else:
    print("CUDA is not available. PyTorch will use the CPU.")

CUDA is available! Here are the GPU details:
  Number of GPUs: 1
  Current GPU device: 0
  GPU Name: Tesla T4
  CUDA Version: 12.8
  Total Memory: 14.56 GB
  Compute Capability: 7.5


In [3]:
# Install qwen tts package

!pip install qwen-tts

In [ ]:
# According to README:
# "Additionally, we recommend using FlashAttention 2 to reduce GPU memory usage.""

!pip install -U flash-attn --no-build-isolation

In [4]:
# Proxy localhost port 8000 to external network (should not be required any more)

from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8000)"))

https://8000-gpu-t4-s-zj0eypnzpope-c.asia-southeast1-1.prod.colab.dev


In [5]:
# Run qwen tts server with 0.6B model

!qwen-tts-demo Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice --ip 0.0.0.0 --port 8000 --share

2026-03-02 20:09:14.970484: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772482154.995187    8030 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772482155.002921    8030 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772482155.022090    8030 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772482155.022115    8030 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772482155.022119    8030 computation_placer.cc:177] computation placer alr

In [6]:
# Voice Design (english)

import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

# single inference
wavs, sr = model.generate_voice_design(
    text="This is a short test using english language.",
    language="English",
    instruct="Young male speaker with a happy tone",
)
sf.write("output_voice_design.wav", wavs[0], sr)


********
********
 


ImportError: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.

In [ ]:
# Voice Design (german)

import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

# single inference
wavs, sr = model.generate_voice_design(
    text="Das ist nur ein kleiner Test.",
    language="German",
    instruct="Old female speaker with a happy tone",
)
sf.write("output_voice_design-de.wav", wavs[0], sr)

In [ ]:
# Voice clone (english)

import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

ref_audio = "https://thorsten-voice.de/ref/ref-en.wav"
ref_text  = "This is a simple reference phrase for voice cloning tests in multiple open source text to speech projects."

wavs, sr = model.generate_voice_clone(
    text="I am curious how close this cloned voice will be compared to my original reference phrase.",
    language="English",
    ref_audio=ref_audio,
    ref_text=ref_text,
)
sf.write("output_voice_clone.wav", wavs[0], sr)

In [ ]:
# Voice clone (german)

import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

ref_audio = "https://thorsten-voice.de/ref/ref-de.wav"
ref_text  = "Dies ist ein einfacher Referenzsatz für das Klonen von Stimmen in unterschiedlichen freien Sprachsynthese Projekten."

wavs, sr = model.generate_voice_clone(
    text="Ich bin sehr gespannt, wie gut die Ähnlichkeit meiner geklonten KI Stimme im Vergleich zur echten Aufnahme ist.",
    language="German",
    ref_audio=ref_audio,
    ref_text=ref_text,
)
sf.write("output_voice_clone-de.wav", wavs[0], sr)

### ALIBABA

https://modelstudio.console.alibabacloud.com/ap-southeast-1?tab=doc&accounttraceid=cf6a742814fb47d8925a5bca70b9bb8ezpsq#/doc/?type=model&url=2840914_2&modelId=qwen3-tts-flash

https://qwen.ai/apiplatform

https://www.alibabacloud.com/help/en/model-studio/qwen-tts?spm=a2c63.p38356.help-menu-2400256.d_0_3_2_2.2b37742emUVw4P#9499fdcba8g88

https://home.console.alibabacloud.com/home/dashboard/ProductAndService



In [11]:
!pip install dashscope

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 71.6 MB/s eta 0:00:00


In [46]:
import os
import dashscope
from google.colab import userdata # Importa la función para acceder a los secretos de Colab

# Obtiene la clave de API del Gestor de Secretos
dashscope_api_key = userdata.get('DASHSCOPE_API_KEY')

# This is the URL for the Singapore region. If you use a model in the China (Beijing) region, replace the URL with: https://dashscope.aliyuncs.com/api/v1
dashscope.base_http_api_url = 'https://dashscope-intl.aliyuncs.com/api/v1'

text = "¡Hola! Qué gusto saludarte. Hoy es un día maravilloso para crear proyectos de inteligencia artificial." #"Today is a wonderful day to build something people love!"
# To use the SpeechSynthesizer interface: dashscope.audio.qwen_tts.SpeechSynthesizer.call(...)
response = dashscope.MultiModalConversation.call(
    # Replace the model with qwen3-tts-instruct-flash to use instruction control.
    model="qwen3-tts-flash",
    # API keys for the Singapore and China (Beijing) regions are different. To get an API key: https://www.alibabacloud.com/help/zh/model-studio/get-api-key
    # If you have not configured an environment variable, replace the following line with your Model Studio API key: api_key = "sk-xxx"
    api_key= dashscope_api_key, #os.getenv("DASHSCOPE_API_KEY"),
    text=text,
    voice="Serena", # Prueba cambiar a "Ryan", "Cherry", "Eric" o "Serena", "VIvian"

    #language_type="English", # We recommend that this matches the text language to ensure correct pronunciation and natural intonation.
    # To use instruction control, uncomment the following lines and replace the model with qwen3-tts-instruct-flash.
    # instructions='Speak quickly with a noticeable rising intonation, suitable for introducing fashion products.',
    # optimize_instructions=True,
    language_type="Spanish",
    stream=False
)
print(response)


{"status_code": 200, "request_id": "789995d6-ad7e-929a-afc3-62561197e7f3", "code": "", "message": "", "output": {"text": null, "finish_reason": "stop", "choices": null, "audio": {"data": "", "url": "http://dashscope-result-sgp.oss-ap-southeast-1.aliyuncs.com/1d/78/20260303/ec611bc6/2fd6b70a-9236-4f15-a41d-b011f7e795fc.wav?Expires=1772583707&OSSAccessKeyId=LTAI5tBLUzt9WaK89DU8aECd&Signature=U%2FwdRYlrhQJcDav3JuomQLzYRwk%3D", "id": "audio_789995d6-ad7e-929a-afc3-62561197e7f3", "expires_at": 1772583707}}, "usage": {"input_tokens": 0, "output_tokens": 0, "characters": 102}}


In [47]:
from IPython.display import display, Audio

# Verificar que la llamada haya sido exitosa (status 200)
if response.status_code == 200:
    # 1. Extraer la URL de donde se encuentra alojado el audio generado
    audio_url = response.output.audio.url

    # 2. Descargar el audio a la máquina virtual de Colab
    # (Hacerlo así evita posibles errores de CORS si intentas reproducir la URL directamente)
    audio_response = requests.get(audio_url)
    archivo_salida = "audio_generado.wav"

    with open(archivo_salida, 'wb') as f:
        f.write(audio_response.content)

    print("¡Audio generado y descargado con éxito!")

    # 3. Mostrar el reproductor nativo de Google Colab
    display(Audio(filename=archivo_salida, autoplay=True))

else:
    # Manejo de errores por si la API falla
    print(f"Error en la petición: Código {response.status_code}")
    print(response.message)

¡Audio generado y descargado con éxito!


### Otra API que tiene modeos muy buenos pero un poco Caros

https://www.mulerouter.ai/docs/api-reference/endpoint/midjourney/diffusion

https://www.mulerouter.ai/app/tasks

In [27]:
import requests

url = "https://api.mulerouter.ai/vendors/google/v1/nano-banana-2/generation"

payload = {
    "prompt": "A photorealistic close-up portrait of an elderly Japanese ceramicist with a warm, knowing smile",
    "aspect_ratio": "3:4",
    "resolution": "2K"
}
headers = {
    "Authorization": "Bearer",
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

print(response.text)

{"task_info":{"id":"633ef309-e550-472e-a661-779edcc31b75","status":"pending","created_at":"2026-03-02T21:26:17.466293+00:00","updated_at":"2026-03-02T21:26:17.466293+00:00"}}


In [28]:
import requests

url = "https://api.mulerouter.ai/vendors/google/v1/nano-banana-2/generation/633ef309-e550-472e-a661-779edcc31b75"

headers = {"Authorization": "Beare "}

response = requests.get(url, headers=headers)

print(response.text)

{"task_info":{"id":"633ef309-e550-472e-a661-779edcc31b75","status":"completed","created_at":"2026-03-02T21:26:17.466293+00:00","updated_at":"2026-03-02T21:26:58.453143+00:00"},"images":["https://mule-router-assets.muleusercontent.com/router_public/production/ephemeral/633ef309-e550-472e-a661-779edcc31b75/result_00.png"],"description":"Here is the photorealistic close-up portrait of the elderly Japanese ceramicist, capturing a moment of warmth and experience."}
